# Week 6 Session 1
## Spark on Databricks Free Edition: Bus Operation Overview
Goal - understand the workspace, notebook, serverless compute

In [0]:
info_df = spark.sql("""
                    SELECT current_catalog() as catalog, 
                    current_schema() as schema
                    """
)

bdisplay(info_df)


In [0]:
print("Spark version: ", spark.version)

In [0]:
%sql
SELECT current_catalog() AS catalog, current_schema() AS schema

In [0]:
bus_ops_data = [
    {"service_date": "2026-04-01", "route_id": "12",  "borough": "Central", "scheduled_trips": 120, "completed_trips": 116, "avg_delay_min": 3.2},
    {"service_date": "2026-04-01", "route_id": "24",  "borough": "Central", "scheduled_trips": 98,  "completed_trips": 92,  "avg_delay_min": 4.8},
    {"service_date": "2026-04-01", "route_id": "73",  "borough": "North",   "scheduled_trips": 110, "completed_trips": 101, "avg_delay_min": 6.1},
    {"service_date": "2026-04-01", "route_id": "149", "borough": "North",   "scheduled_trips": 105, "completed_trips": 96,  "avg_delay_min": 5.7},
    {"service_date": "2026-04-01", "route_id": "205", "borough": "East",    "scheduled_trips": 88,  "completed_trips": 80,  "avg_delay_min": 7.4},
    {"service_date": "2026-04-01", "route_id": "253", "borough": "East",    "scheduled_trips": 90,  "completed_trips": 83,  "avg_delay_min": 6.9},
    {"service_date": "2026-04-01", "route_id": "11",  "borough": "West",    "scheduled_trips": 95,  "completed_trips": 91,  "avg_delay_min": 2.9},
    {"service_date": "2026-04-01", "route_id": "23",  "borough": "West",    "scheduled_trips": 102, "completed_trips": 99,  "avg_delay_min": 3.5}
]

bus_df = spark.createDataFrame(bus_ops_data)

display(bus_df)

In [0]:
bus_df.printSchema()
print("Total rows: ", bus_df.count())

In [0]:
from pyspark.sql.functions import col, round

bus_kpi_df = (
    bus_df
    .withColumn("completion_rate", round(col("completed_trips") / col("scheduled_trips"), 3))
)

display(bus_kpi_df)
    

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, round

borough_summaries_df = (
    bus_kpi_df
    .groupBy("borough")
    .agg(
        spark_sum("scheduled_trips").alias("total_scheduled_trips"),
        spark_sum("completed_trips").alias("total_completed_trips"),
        round(avg("avg_delay_min"), 2).alias("avg_delay_min"),
    )
    .orderBy("borough")
)
display(borough_summaries_df)